# Notebook 03 — Normalization and Log Transformation

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 03 of 12  
**Time estimate:** 75 minutes

> Two cells expressing the same gene at the same level will appear different if one
> was sequenced 3× deeper than the other.
> Normalization removes technical library-size variation before biological variation is analyzed.

---
## Step 1 — Motivation

Raw UMI counts are proportional to both (1) true gene expression and (2) how many
mRNA molecules were captured per cell (library size). Cells with more total UMIs will
have higher counts for every gene — even genes expressed identically across cells.
Normalization corrects for this so that count differences reflect biology, not depth.

---
## Step 2 — Intuition

**Library-size normalization:** Scale each cell's counts so all cells have the same
total. Convention: target = 10,000 ("counts per 10k", CP10K). After normalization,
each gene's value represents "UMIs per 10,000 total UMIs" — comparable across cells.

**Log1p transformation:** After normalization, expression values span many orders of
magnitude (0 to thousands). Log-transforming compresses this range, making variance
more homogeneous (heteroscedasticity → homoscedasticity). We use `log(x + 1)` (log1p)
rather than `log(x)` to handle the many zero values.

**Highly Variable Genes (HVGs):** Most genes don't vary much across cells — they're
housekeeping genes expressed similarly everywhere. Keeping only the top ~2000 HVGs
focuses PCA and clustering on the genes that actually discriminate cell types.

---
## Step 3 — Biological Background

**Why CP10K (10,000) not CPM (1,000,000)?**  
Typical scRNA-seq cells have 1,000–5,000 UMIs. Normalizing to 10,000 keeps most
normalized values in an intuitive range (1–1000). CPM (scaling to 1,000,000) was
designed for bulk RNA-seq with millions of reads per sample.

**This is NOT TPM:**  
In scRNA-seq, we do not divide by gene length. UMI counts already represent individual
mRNA molecules — the 3' capture approach means all mRNAs from a gene are equally
likely to be captured regardless of gene length. Gene length correction applies to
read-based bulk RNA-seq (RPKM/TPM) where longer genes generate more reads.

**Highly Variable Genes:**  
A gene is "highly variable" if its expression level genuinely differs between cell
types. The standard approach identifies genes where observed variance exceeds the
variance expected from Poisson noise alone — a "normalized dispersion" score.

---
## Step 4 — Mathematical Explanation

**Library-size normalization:**
$$x_{ij}^{\text{norm}} = \frac{x_{ij}}{s_i} \times 10{,}000$$

where $s_i = \sum_j x_{ij}$ is cell $i$'s library size.

**Log1p transformation:**
$$x_{ij}^{\text{log}} = \log(x_{ij}^{\text{norm}} + 1)$$

After this, $x_{ij}^{\text{log}} \approx 0$ for unexpressed genes and up to ~8–9
for highly expressed genes (since $\log(10000 + 1) \approx 9.2$ for a gene
accounting for all UMIs).

**HVG selection via normalized dispersion:**  
Group genes into bins by mean expression. Within each bin, compute:
$$\text{normalized\_dispersion}(g) = \frac{\text{dispersion}(g) - \text{mean\_bin\_dispersion}}{\text{std\_bin\_dispersion}}$$

where $\text{dispersion}(g) = \text{variance}(g) / \text{mean}(g)$ (coefficient of
variation squared, analogous to the Fano factor). Genes with high normalized dispersion
vary more than their expression-level peers — those are the biologically informative genes.

In [ ]:
# Step 6 — Python: Normalization, log1p, and HVG selection from scratch
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# ---- Simulate post-QC count matrix ----
N_CELLS = 250
N_GENES = 1200
N_MITO = 13
cell_types = rng.choice(['T_cell', 'B_cell', 'Monocyte'], N_CELLS, p=[0.4, 0.35, 0.25])

def sim_clean_matrix(n_cells, n_genes, cell_types, rng):
    X = np.zeros((n_cells, n_genes), dtype=np.int32)
    markers = {
        'T_cell':   (list(range(0, 40)), 5.0),
        'B_cell':   (list(range(40, 80)), 6.0),
        'Monocyte': (list(range(80, 130)), 4.0),
    }
    for i, ct in enumerate(cell_types):
        mg, boost = markers[ct]
        base = rng.exponential(0.3, n_genes)
        base[mg] *= boost
        lib = int(np.clip(rng.normal(2500, 500), 800, 7000))
        X[i, :] = rng.multinomial(lib, base / base.sum())
    return X

X_raw = sim_clean_matrix(N_CELLS, N_GENES, cell_types, rng)

# ---- 1. Library-size normalization to CP10K ----
def normalize_total(X, target=1e4):
    """Scale each cell to target total counts."""
    lib_sizes = X.sum(axis=1, keepdims=True).astype(float)
    return X.astype(float) / lib_sizes * target

X_norm = normalize_total(X_raw, target=1e4)

# Verify: all cells should now sum to 10,000
assert np.allclose(X_norm.sum(axis=1), 1e4, atol=0.01), 'Normalization failed'
print(f'All cells sum to 10,000: {np.allclose(X_norm.sum(axis=1), 1e4, atol=0.01)}')

# ---- 2. Log1p transformation ----
X_log = np.log1p(X_norm)
print(f'Post-log value range: [{X_log.min():.2f}, {X_log.max():.2f}]')

# ---- 3. HVG selection via normalized dispersion ----
def select_hvgs(X_log, n_top_genes=200, n_bins=20):
    """Select highly variable genes by normalized dispersion."""
    means = X_log.mean(axis=0)         # shape (n_genes,)
    variances = X_log.var(axis=0)      # shape (n_genes,)
    # Avoid division by zero
    dispersions = np.where(means > 0, variances / (means + 1e-9), 0.0)

    # Bin genes by mean expression
    bin_edges = np.percentile(means, np.linspace(0, 100, n_bins + 1))
    gene_bins = np.digitize(means, bin_edges[:-1]) - 1
    gene_bins = np.clip(gene_bins, 0, n_bins - 1)

    norm_disp = np.zeros_like(dispersions)
    for b in range(n_bins):
        mask = gene_bins == b
        if mask.sum() < 2:
            continue
        bin_disp = dispersions[mask]
        bin_mean = bin_disp.mean()
        bin_std = bin_disp.std() + 1e-9
        norm_disp[mask] = (bin_disp - bin_mean) / bin_std

    # Select top n_top_genes by normalized dispersion
    top_idx = np.argsort(norm_disp)[::-1][:n_top_genes]
    hvg_mask = np.zeros(X_log.shape[1], dtype=bool)
    hvg_mask[top_idx] = True

    return hvg_mask, means, dispersions, norm_disp

hvg_mask, means, dispersions, norm_disp = select_hvgs(X_log, n_top_genes=200)
X_hvg = X_log[:, hvg_mask]

print(f'HVGs selected: {hvg_mask.sum()} / {N_GENES}')
print(f'Data shape after HVG selection: {X_hvg.shape}')

# Check HVG overlap with known marker genes (indices 0–130)
marker_indices = set(range(0, 130))
hvg_indices = set(np.where(hvg_mask)[0])
overlap = len(marker_indices & hvg_indices)
print(f'Marker genes recovered in HVGs: {overlap}/130 ({overlap/130*100:.0f}%)')

In [ ]:
# Step 7 — Visualization: Before/after normalization and HVG selection
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Panel A: Library size before normalization
ax = axes[0, 0]
lib_sizes_raw = X_raw.sum(axis=1)
ax.hist(lib_sizes_raw, bins=40, color='steelblue', alpha=0.8)
ax.set_xlabel('Total UMI counts')
ax.set_ylabel('Number of cells')
ax.set_title('A. Library size distribution\n(before normalization)')

# Panel B: Single gene, raw vs normalized
ax = axes[0, 1]
gene_idx = 0  # A T-cell marker gene
ax.scatter(lib_sizes_raw, X_raw[:, gene_idx], alpha=0.4, s=8, label='raw counts')
ax.set_xlabel('Library size (total UMIs)')
ax.set_ylabel(f'Gene {gene_idx} counts')
ax.set_title('B. Raw counts correlate with library size')

# Panel C: Same gene after normalization
ax = axes[0, 2]
ax.scatter(lib_sizes_raw, X_norm[:, gene_idx], alpha=0.4, s=8,
            color='tomato', label='normalized')
ax.set_xlabel('Library size (total UMIs)')
ax.set_ylabel(f'Gene {gene_idx} normalized')
ax.set_title('C. Normalized counts: library size removed')

# Panel D: Effect of log1p — distribution shape
ax = axes[1, 0]
flat_norm = X_norm.flatten()
flat_log = X_log.flatten()
flat_norm_nz = flat_norm[flat_norm > 0]
flat_log_nz = flat_log[flat_log > 0]
ax.hist(flat_norm_nz, bins=60, density=True, alpha=0.6, label='CP10K (non-zero)', color='steelblue')
ax2t = ax.twinx()
ax2t.hist(flat_log_nz, bins=60, density=True, alpha=0.4, label='log1p (non-zero)', color='tomato')
ax.set_xlabel('Value')
ax.set_ylabel('Density (CP10K)', color='steelblue')
ax2t.set_ylabel('Density (log1p)', color='tomato')
ax.set_title('D. CP10K vs. log1p: distribution compression')

# Panel E: Mean-variance relationship
ax = axes[1, 1]
hvg_colors = np.where(hvg_mask, 'tomato', 'lightblue')
ax.scatter(means, dispersions, c=hvg_colors, s=5, alpha=0.5)
ax.set_xlabel('Mean log1p expression')
ax.set_ylabel('Dispersion (var/mean)')
ax.set_title('E. Mean-dispersion plot\n(red = selected HVGs)')

# Panel F: Normalized dispersion ranks
ax = axes[1, 2]
rank_order = np.argsort(norm_disp)[::-1]
ax.plot(norm_disp[rank_order], lw=1, color='steelblue')
ax.axvline(200, color='red', ls='--', label='HVG cutoff (top 200)')
ax.set_xlabel('Gene rank (by norm. dispersion)')
ax.set_ylabel('Normalized dispersion')
ax.set_title('F. Normalized dispersion — elbow for HVG selection')
ax.legend(fontsize=8)

plt.suptitle('Module 10 NB03: Normalization and HVG Selection', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scrna_normalization.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. After CP10K normalization, verify that all cells sum to exactly 10,000.
2. What is the log1p value for a gene with 0 counts? For a gene with 10,000 counts
   (i.e., accounting for all UMIs in a cell)?
3. Increase `n_top_genes` to 500 HVGs. How does the marker gene recovery rate change?
4. Why do we normalize before computing HVGs rather than after?

---
## Step 10 — Quiz

1. What does CP10K stand for and what does the normalization do to each cell?
2. Why is log1p used rather than `log(x)` in scRNA-seq?
3. Why is gene-length normalization (TPM) NOT applied in scRNA-seq?
4. What is the "dispersion" of a gene and why is a high dispersion informative?
5. What is the purpose of binning genes by mean expression before computing
   normalized dispersion?

---
## Step 12 — Reflection

> *[In one paragraph: explain why normalization cannot perfectly remove
> all technical variation, and name at least two sources of technical variation
> that persist after CP10K + log1p normalization.]*

---
*Next: `04_pca_for_single_cell.ipynb`*